# raw

In [0]:
raw_file=spark.read.format("csv").option("header","true").option("inferSchema","true").load('abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/raw/customer')

In [0]:
raw_file.display()

cust_id,cust_name,address
C1,srinivas,bangalore
C2,Phani,Chennai
C3,Ram,Hyderabad
C4,Krishna,Hyderabad


In [0]:

raw_file.write.format("delta").mode("overwrite").saveAsTable("shivasai_catalog.bronze.customer_stage")

In [0]:
%sql
select * from shivasai_catalog.bronze.customer_stage

cust_id,cust_name,address
C1,srinivas,bangalore
C2,Phani,Chennai
C3,Ram,Hyderabad
C4,Krishna,Hyderabad


In [0]:
%sql
create table if not exists shivasai_catalog.silver.customer_target
(
    cust_id string,
    cust_name string,
    address string
)

using delta

In [0]:
%sql
merge into shivasai_catalog.silver.customer_target
using shivasai_catalog.bronze.customer_stage
on customer_target.cust_id = customer_stage.cust_id
when matched then update set *
when not matched then insert *


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,4,0,0


In [0]:
%sql
select * from shivasai_catalog.silver.customer_target

cust_id,cust_name,address
C1,srinivas,bangalore
C2,Phani,Chennai
C3,Ram,Hyderabad
C4,Krishna,Hyderabad


In [0]:
dbutils.fs.mkdirs("abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/archiev")


True

In [0]:
files_list = dbutils.fs.ls("abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/raw/customer")

for files in files_list:
    print(files.path)
    dbutils.fs.mv(files.path, "abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/archiev")

abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/raw/customer/cust_01062026.csv


In [0]:
dbutils.fs.ls("abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/archiev")

[FileInfo(path='abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/archiev/cust_01052026.csv', name='cust_01052026.csv', size=80, modificationTime=1794447168000),
 FileInfo(path='abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/archiev/cust_01062026.csv', name='cust_01062026.csv', size=108, modificationTime=1794447267000)]